# 8.1 世界模型概念介绍

> **环境准备**：本节为纯理论+代码演示，不需要 AirSim 模拟器。只需要 PyTorch。
>
> **AirSim 配置说明**：后续实验（8.3~8.6节）需要 AirSim。请将本目录下的 `settings.json` 复制到 `~/Documents/AirSim/settings.json`，然后启动 AirSim。该配置为单架无人机（Drone1），相机分辨率 64×64（适配世界模型输入）。
>
> **Python 环境**：使用 `conda activate drone` 环境，需安装 `pip install torch gymnasium`。

## 学习目标

- 理解从被动决策到主动预测的演进
- 掌握世界模型的核心公式：S + A → S'
- 区分模型强化学习与无模型强化学习
- 理解隐空间建模的必要性
- 动手实现一个最简单的世界模型

## 8.1.1 为什么需要世界模型？

想象你在开车。当你看到前方路口变黄灯时，你会**预判**：
- 如果我加速，能在红灯前通过吗？
- 如果我刹车，后面的车会不会追尾？

你并没有真的去试，而是在**脑中模拟**了不同动作的后果，然后选择最优方案。

这就是世界模型的核心思想：**在脑中建立一个环境的模拟器，用它来预测不同动作的后果，从而做出更好的决策。**

### 传统方法的局限

| 方法 | 决策方式 | 类比 |
|------|---------|------|
| 视觉导航 (VLN) | 看到什么就做什么，不预判未来 | 只看脚下走路 |
| 无模型强化学习 | 反复试错，积累经验 | 摔了100次才学会骑车 |
| **世界模型** | **先在脑中模拟，再行动** | **想象骑车的感觉，少摔几次** |

对于无人机来说，"反复试错"的代价极高——每次坠机都意味着硬件损坏。世界模型让无人机可以在"虚拟想象"中试错，大幅降低真实世界的探索成本。

![从VLN到世界模型的演进](figures/wm_evolution.png)

*图 8-1：从被动反应到主动预测——世界模型让智能体具备了"预判未来"的能力*

## 8.1.2 世界模型的核心公式

世界模型回答一个核心问题：

> **如果我现在处于状态 S，执行动作 A，那么下一时刻的状态 S' 会是什么？**

用数学表示：

$$S_{t+1}, R_t = f(S_t, A_t)$$

其中：
- $S_t$：当前状态（无人机的位置、速度、姿态等）
- $A_t$：当前动作（油门、俯仰、横滚等控制量）
- $S_{t+1}$：预测的下一状态
- $R_t$：预测的奖励（离目标更近了？还是撞墙了？）
- $f$：世界模型（一个神经网络）

### 无人机场景的具体例子

| 输入 | 含义 | 示例 |
|------|------|------|
| $S_t$ | 当前状态 | 位置(0,0,-5), 速度(1,0,0), 姿态(水平) |
| $A_t$ | 执行动作 | 向左倾斜30度 |
| $S_{t+1}$ | 预测下一状态 | 位置(-0.5,0,-5), 速度(0.5,-1,0), 姿态(左倾) |
| $R_t$ | 预测奖励 | +0.8（仍在目标附近） |

![世界模型核心公式](figures/wm_core_formula.png)

*图 8-2：世界模型的核心问题——给定当前状态和动作，预测下一状态和奖励*

## 8.1.3 模型强化学习 vs 无模型强化学习

| 对比 | 无模型 RL (Model-Free) | 模型 RL (Model-Based) |
|------|----------------------|---------------------|
| 代表算法 | DQN, PPO, A2C | DreamerV3, World Models |
| 工作方式 | 在真实环境中反复试错 | 先学世界模型，再在模型中规划 |
| 样本效率 | 低（需要海量交互数据） | 高（可在虚拟世界无限试错） |
| 训练成本 | 高（每次都要真实交互） | 低（大部分训练在想象中完成） |
| 风险 | 高（真实试错可能损坏设备） | 低（虚拟试错零成本） |
| 缺点 | 简单直接，无需建模 | 存在模型偏差风险 |

**模型偏差（Model Bias）**是模型强化学习的主要风险：如果世界模型学得不准确，智能体在"想象"中找到的好策略，在真实世界中可能完全失效。这就像一个人在梦里学会了飞，醒来后发现自己并不会飞。

## 8.1.4 隐空间建模：为什么不直接预测像素？

一个自然的想法是：让世界模型直接预测下一帧图像的每个像素。但这有三个严重问题：

**1. 维度灾难**

一张 64×64 的 RGB 图像有 64×64×3 = 12,288 个像素值。预测这么多数值的难度远大于预测 6 个状态变量（位置xyz + 速度xyz）。

**2. 信息冗余**

图像中大量像素对决策毫无帮助——天空的颜色、地面的纹理、远处的云彩，这些都不影响无人机的飞行决策。

**3. 噪声敏感**

光照变化、传感器噪声等微小的像素级变化会严重干扰预测。

### 解决方案：隐空间

```
高维图像 (12288维) → [编码器] → 隐状态 (32维) → [世界模型] → 预测隐状态 → [解码器] → 预测图像
```

编码器将图像压缩为低维的"隐状态向量"，只保留对决策有用的核心信息。世界模型在这个低维空间中进行预测，效率提升数百倍。

## 8.1.5 动手实现：最简单的世界模型

下面我们用 PyTorch 实现一个最简单的世界模型：一个 MLP（多层感知机）网络，输入当前状态和动作，预测下一状态和奖励。

虽然真正的 DreamerV3 要复杂得多，但核心思想完全一致：**学习 S + A → S' 的映射关系**。

In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# 定义最简单的世界模型
class SimpleWorldModel(nn.Module):
    def __init__(self, state_dim=6, action_dim=4, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.state_head = nn.Linear(hidden_dim, state_dim)
        self.reward_head = nn.Linear(hidden_dim, 1)

    def forward(self, state, action):
        x = torch.cat([state, action], dim=-1)
        h = self.net(x)
        return self.state_head(h), self.reward_head(h)

model = SimpleWorldModel()
print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")
print(f"输入: 状态(6维) + 动作(4维) = 10维")
print(f"输出: 下一状态(6维) + 奖励(1维) = 7维")

模型参数量: 5,319
输入: 状态(6维) + 动作(4维) = 10维
输出: 下一状态(6维) + 奖励(1维) = 7维


### 生成模拟数据并训练

我们用一个简单的物理规则生成训练数据：无人机在二维平面上运动，动作控制加速度。

In [2]:
# 生成模拟数据：简单的二维运动
def generate_data(n_episodes=200, steps_per_episode=50):
    states, actions, next_states, rewards = [], [], [], []
    for _ in range(n_episodes):
        pos = np.random.randn(3) * 2  # 初始位置
        vel = np.zeros(3)              # 初始速度
        for _ in range(steps_per_episode):
            state = np.concatenate([pos, vel]).astype(np.float32)
            action = np.random.randn(4).astype(np.float32) * 0.5
            # 简单物理：动作前3维控制加速度，第4维控制垂直推力
            acc = action[:3] * 0.1
            acc[2] += action[3] * 0.05 - 0.02  # 重力补偿
            vel = vel * 0.95 + acc  # 阻尼
            pos = pos + vel * 0.1
            next_state = np.concatenate([pos, vel]).astype(np.float32)
            reward = max(0, 1.0 - np.linalg.norm(pos) / 5.0)  # 靠近原点奖励高
            states.append(state)
            actions.append(action)
            next_states.append(next_state)
            rewards.append([reward])
    return (torch.tensor(np.array(x)) for x in [states, actions, next_states, rewards])

states, actions, next_states, rewards = generate_data()
print(f"训练数据: {states.shape[0]} 个样本")
print(f"状态维度: {states.shape[1]}, 动作维度: {actions.shape[1]}")

训练数据: 10000 个样本
状态维度: 6, 动作维度: 4


In [3]:
# 训练世界模型
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []

for epoch in range(100):
    pred_states, pred_rewards = model(states, actions)
    loss_state = nn.MSELoss()(pred_states, next_states)
    loss_reward = nn.MSELoss()(pred_rewards, rewards)
    loss = loss_state + loss_reward

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('世界模型训练损失')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"最终损失: {losses[-1]:.6f}")

RuntimeError: Found dtype Double but expected Float

### 测试：用世界模型"想象"未来轨迹

![悬停与避障任务对比](figures/hover_vs_avoidance.png)

*图 8-3：后续两个实验的任务对比——从简单的悬停控制到复杂的避障决策*

In [ ]:
# 用训练好的世界模型预测未来轨迹
model.eval()
with torch.no_grad():
    # 初始状态：位置(2,2,-3)，速度(0,0,0)
    state = torch.tensor([[2.0, 2.0, -3.0, 0.0, 0.0, 0.0]])
    trajectory = [state.numpy()[0, :3]]

    for step in range(50):
        # 简单策略：朝原点方向施加力
        pos = state[0, :3]
        action = -pos * 0.3  # 朝原点方向
        action = torch.cat([action, torch.tensor([0.2])]).unsqueeze(0)  # 加垂直推力
        state, reward = model(state, action)
        trajectory.append(state.numpy()[0, :3])

trajectory = np.array(trajectory)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# XY平面轨迹
axes[0].plot(trajectory[:, 0], trajectory[:, 1], 'b-o', markersize=3)
axes[0].plot(trajectory[0, 0], trajectory[0, 1], 'go', markersize=10, label='起点')
axes[0].plot(0, 0, 'r*', markersize=15, label='目标(原点)')
axes[0].set_xlabel('X'); axes[0].set_ylabel('Y')
axes[0].set_title('世界模型"想象"的飞行轨迹 (XY平面)')
axes[0].legend(); axes[0].grid(True, alpha=0.3); axes[0].set_aspect('equal')

# 高度变化
axes[1].plot(trajectory[:, 2], 'r-o', markersize=3)
axes[1].axhline(y=0, color='k', linestyle='--', alpha=0.3, label='地面')
axes[1].set_xlabel('时间步'); axes[1].set_ylabel('Z (高度)')
axes[1].set_title('高度变化'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"起点: ({trajectory[0,0]:.1f}, {trajectory[0,1]:.1f}, {trajectory[0,2]:.1f})")
print(f"终点: ({trajectory[-1,0]:.1f}, {trajectory[-1,1]:.1f}, {trajectory[-1,2]:.1f})")

## 8.1.6 小结

本节的核心要点：

1. **世界模型**让智能体能够在"脑中"模拟环境，预判不同动作的后果
2. 核心公式：$S_{t+1}, R_t = f(S_t, A_t)$
3. **模型强化学习**比无模型方法样本效率高得多，特别适合无人机等试错成本高的场景
4. **隐空间建模**将高维图像压缩为低维向量，让预测变得高效
5. 即使是最简单的 MLP 世界模型，也能学会预测物理运动规律

下一节，我们将深入 DreamerV3——当前最先进的世界模型架构之一。